# 01b · Extra insights for Part 1 (bonus analyses)

**Project 2 · Problem 2 · Part 1.** `01_eda.ipynb` answers the six required questions. This notebook adds **four extra analyses**, each attached to one of those questions, and each ends with the *story*: what we found, why it matters and which decision it supports.

| # | Extends | Question we ask | Decision it supports |
|---|---------|-----------------|----------------------|
| 1 | Q1 (distribution of `overall`) | Does the rating mix drift over time, and does the **test set look like the training set**? | Keep the natural class distribution; trust the validation score |
| 2 | Q3 (top reviewers) | How concentrated are helpful votes? | `vote` is a popularity signal, not a quality label |
| 3 | Q5 (top products) | Is a product with many 5-star reviews also *loved*? | Count of 5-stars measures popularity; use the **share** for quality |
| 4 | Q6 (top brands) | Are the brand differences statistically real? | Which brand rankings we can defend |

Run from the `notebooks/` folder. Loading the full training set with only a few columns takes about a minute.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `src` importable when running from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)

In [ ]:
train = data.load_train(columns=["overall", "vote", "verified", "unixReviewTime", "reviewerID", "asin", "reviewText"])
test = data.load_test(columns=["vote", "verified", "unixReviewTime", "reviewText"])
products = data.load_products()

for frame in (train, test):
    frame["year"] = pd.to_datetime(frame["unixReviewTime"], unit="s").dt.year
    frame["n_chars"] = frame["reviewText"].fillna("").str.len()
print(f"train {len(train):,} reviews | test {len(test):,} reviews | years {train['year'].min()}-{train['year'].max()}")

## 1 · Q1 extension — the rating mix over time, and is the test set like the training set?

In [ ]:
by_year = train.groupby("year").agg(
    n_reviews=("overall", "size"),
    mean_rating=("overall", "mean"),
    five_star_pct=("overall", lambda s: (s == 5).mean() * 100),
    one_star_pct=("overall", lambda s: (s == 1).mean() * 100),
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].bar(by_year.index, by_year["n_reviews"], color="#95a5a6")
axes[0].set(title="Reviews per year", xlabel="year", ylabel="reviews")
axes[1].plot(by_year.index, by_year["five_star_pct"], marker="o", label="5-star share (%)", color="#27ae60")
axes[1].plot(by_year.index, by_year["one_star_pct"], marker="o", label="1-star share (%)", color="#c0392b")
axes[1].set(title="Share of extreme ratings over time", xlabel="year", ylabel="% of reviews")
axes[1].legend()
plt.tight_layout(); plt.show()
by_year

**Story.** All reviews are from **2016–2018**, and volume falls each year (the training file is dominated by 2016). Ratings drift *slightly* downwards: the 5-star share goes from **55.8 % (2016)** to **53.5 % (2018)** and the 1-star share from **9.0 % to 11.5 %**. The drift is small compared with the 55 % vs 7 % class imbalance, so it does not change our modelling plan, but it tells us not to expect the class distribution to be perfectly stable.

In [ ]:
# Does the (unlabeled) test set look like the training set? Compare everything we can see without labels.
def profile(frame):
    return pd.Series({
        "verified share (%)": frame["verified"].mean() * 100,
        "reviews with votes (%)": (frame["vote"] > 0).mean() * 100,
        "median characters": frame["n_chars"].median(),
        "mean year": frame["year"].mean(),
    })

shift = pd.DataFrame({"train": profile(train), "test": profile(test)}).round(2)
display(shift)

year_share = pd.concat({"train": train["year"].value_counts(normalize=True),
                        "test": test["year"].value_counts(normalize=True)}, axis=1).sort_index().mul(100).round(1)
year_share.plot(kind="bar", figsize=(10, 3.8), rot=0, title="Share of reviews per year (%)")
plt.tight_layout(); plt.show()

**Story.** The test set has no labels, so the best we can do is compare what we *can* see. Verified share (85.9 % vs 85.6 %) and the mean year (2016.57 vs 2016.59) are practically identical, so the test reviews come from the same period and the same kind of buyers: **there is no sign of a time shift**, which supports our decision *not* to re-balance the training classes (the metric is evaluated on the test distribution).

One honest caveat: the test set has **slightly more reviews with votes (24.7 % vs 22.8 %)** and slightly longer texts (median 424 vs 408 characters). With 20,000 test reviews the vote difference is larger than sampling noise would explain (≈ 0.3 pp), so the test sample is not a perfect copy of training. We cannot tell from the features whether the ratings differ, therefore (a) the validation score is an *estimate*, not a promise, and (b) we should not lean on `vote` or length as model features.

## 2 · Q3 extension — how concentrated are helpful votes?

In [ ]:
# How concentrated are helpful votes?
review_votes = train["vote"].sort_values(ascending=False).to_numpy()
total_votes = review_votes.sum()
cum = review_votes.cumsum() / total_votes
n = len(review_votes)
for pct in [0.1, 1, 5]:
    k = int(n * pct / 100)
    print(f"top {pct:>4}% of reviews ({k:>7,}) hold {cum[k - 1]:.1%} of all votes")
print(f"reviews with at least one vote: {(review_votes > 0).mean():.1%}")

reviewer_votes = train.groupby("reviewerID")["vote"].sum().sort_values(ascending=False)
print(f"top 10 reviewers hold {reviewer_votes.head(10).sum() / total_votes:.2%} of all votes "
      f"({reviewer_votes.head(10).sum():,} of {total_votes:,})")

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(1, n + 1) / n * 100
ax.plot(x[: n // 10], cum[: n // 10] * 100, color="#8e44ad")
ax.set(title="Cumulative share of votes held by the most-voted reviews", xlabel="% of reviews (most voted first)", ylabel="% of all votes")
plt.tight_layout(); plt.show()

**Story.** Helpfulness votes are **extremely concentrated**: the top **1 % of reviews hold 41 %** of all votes, and the top 0.1 % (838 reviews) alone hold **17 %**. At the same time only **22.8 %** of reviews received any vote. The ten reviewers in Q3 together hold just **1.9 %** of all votes (26,885 of 1,444,859): "top reviewer" is a high rank in a very long-tailed distribution, driven by a few viral reviews rather than by a large body of work.

**Decision.** `vote` says how much *attention* a review received, not how good or how positive it is, and it is zero for three quarters of the data. We keep it out of the main model; at most a `log1p(vote)` feature can be tested as an extra.

## 3 · Q5 extension — popular is not the same as loved

In [ ]:
# Popular is not the same as loved: 5-star share of the ten products with the most 5-star reviews.
five = train.loc[train["overall"] == 5].groupby("asin").size().nlargest(10).rename("five_star_reviews")
stats = train[train["asin"].isin(five.index)].groupby("asin")["overall"].agg(total_reviews="size", mean_rating="mean")
top_products = (five.to_frame().join(stats).reset_index()
                .merge(products[["asin", "brand", "title"]], on="asin", how="left"))
top_products["five_star_share_%"] = (top_products["five_star_reviews"] / top_products["total_reviews"] * 100).round(1)
top_products["title"] = top_products["title"].str.slice(0, 55)
top_products["mean_rating"] = top_products["mean_rating"].round(2)
print(f"for reference: {(train['overall'] == 5).mean():.1%} of all reviews are 5-star")
top_products[["brand", "title", "total_reviews", "five_star_reviews", "five_star_share_%", "mean_rating"]]

**Story.** The Q5 ranking counts 5-star reviews, so it rewards products that simply have many reviews. Dividing by each product's total shows how different these "top" products really are. Against a 55 % overall 5-star share:

* The **Samsung 850 EVO SSDs** are genuinely loved: **85–87 %** of their reviews are 5-star (mean 4.7).
* **StarTech USB to SATA adapter** also stands out (75 %).
* **Arlo Wireless Security Camera System** is #9 by 5-star count, but only **38.8 %** of its 1,115 reviews are 5-star and its mean is **3.66**. It is a *popular but polarising* product: many happy buyers and many unhappy ones.

**Decision.** For anything that talks about product quality we use the **share and mean rating**, not the raw count of 5-stars. The count is a measure of popularity.

## 4 · Q6 extension — are the brand differences real?

In [ ]:
# Are the brand differences real? Mean rating with a 95 % confidence interval for the ten most-reviewed brands.
merged = train[["asin", "overall"]].merge(products[["asin", "brand"]], on="asin", how="left").dropna(subset=["brand"])
brand_stats = merged.groupby("brand")["overall"].agg(n="size", mean="mean", std="std")
top = brand_stats.nlargest(10, "n").copy()
top["ci95"] = 1.96 * top["std"] / np.sqrt(top["n"])
top = top.sort_values("mean")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(top["mean"], range(len(top)), xerr=top["ci95"], fmt="o", color="#2c3e50", capsize=3)
ax.axvline(train["overall"].mean(), color="#c0392b", ls="--", lw=1, label="mean of all reviews")
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index)
ax.set(title="Mean rating of the 10 most-reviewed brands (95% CI)", xlabel="mean overall rating")
ax.legend(); plt.tight_layout(); plt.show()
top[["n", "mean", "ci95"]].round(3).sort_values("mean", ascending=False)

**Story.** The Q6 table sorted brands by mean rating. With 7,000–17,000 reviews per brand the 95 % intervals are narrow (±0.02 to ±0.04 stars), which lets us separate groups:

* **NETGEAR (3.81)** is clearly the lowest: its interval does not overlap any other brand.
* **Anker (4.24)** and **AmazonBasics (4.20)** are clearly above the rest, but their two intervals overlap, so we **cannot say that Anker is better than AmazonBasics**.
* The middle (Asus, Sony, TP-LINK, Samsung, Logitech, SanDisk) lies between 3.86 and 3.96, differences of a few hundredths that are mostly within the intervals.

**Caveat.** These brands sell very different product types (routers, cables, storage). A brand's mean rating mixes product quality with category difficulty (routers and networking gear attract more complaints), so it is a description, not a quality ranking.

## Summary

| Analysis | Key finding | Used for |
|----------|-------------|----------|
| Time trend | 5-star share 55.8 % → 53.5 %, 1-star 9.0 % → 11.5 % (2016 → 2018) | Expect mild drift; keep natural distribution |
| Train vs test | Same period and verified share; test has a few more voted reviews | Validation is an estimate; avoid `vote` / length as features |
| Vote concentration | Top 1 % of reviews hold 41 % of votes; 22.8 % of reviews have any | `vote` = attention, not quality |
| Popularity vs love | 5-star share of top products ranges from 39 % to 87 % | Use shares, not counts |
| Brand intervals | NETGEAR lowest; Anker ≈ AmazonBasics highest | Defensible brand statements only |